# vla-knew: does OpenVLA-7B "know" before it acts wrongly? (LIBERO-Goal, linear probes vs output-space gates)
**Branch:** `feature/vla-knew` | **Gate:** pre-registered in README.md (frozen 2026-11-08)
**One-Click Execution:** Runtime -> Run all (L4). Clones, installs LIBERO + pinned OpenVLA deps, runs the GPU pre-flight
(aborts if invalid), runs the resumable campaign with results and the HF cache on Google Drive, prints RESULTS.md,
unassigns the GPU. Re-running after a disconnect resumes from Drive (finished episode groups are skipped).

In [ ]:
# Step 1: Mount Drive; results (incl. resume state) and the 15 GB HF checkpoint cache live there
import os  # first cell: the Drive prompt appears right away
from google.colab import drive
drive.mount("/content/drive")
os.environ["VK_RESULTS"] = "/content/drive/MyDrive/moonshot_results/vla-knew"
os.environ["HF_HOME"] = "/content/hf_cache"
os.makedirs(os.environ["VK_RESULTS"], exist_ok=True)
print("Results dir:", os.environ["VK_RESULTS"], "| existing:", sorted(os.listdir(os.environ["VK_RESULTS"])))

In [ ]:
# Step 2: Credit Guard (auto-unassign on any error or completion) + Colab Secrets
import os, sys, signal, subprocess
from IPython import get_ipython

def auto_teardown(signum=None, frame=None):
    print("\n[Credit Guard] Error, interrupt, or completion! Flushing Drive and unassigning GPU...")
    try:
        from google.colab import drive
        drive.flush_and_unmount()
    except Exception:
        pass
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

def custom_exc_handler(self, etype, value, tb, tb_offset=None):
    self.showtraceback((etype, value, tb), tb_offset=tb_offset)
    auto_teardown()
    return []

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), custom_exc_handler)

try:
    from google.colab import userdata
    for k in ["HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "hf_token"]:
        try:
            tok = userdata.get(k)
            if tok:
                os.environ["HF_TOKEN"] = tok
                os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
                print(f"[Secrets] Loaded Hugging Face token from Colab Secrets ({k}).")
                break
        except Exception:
            pass
except ImportError:
    pass

print("[Ready] Credit guard active. VM will self-terminate on error or completion.")

In [ ]:
# Step 3: GPU check + clone repository
import torch
assert torch.cuda.is_available(), "Switch Colab runtime to GPU (L4 24GB): Runtime -> Change runtime type -> GPU (L4)"
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB | torch {torch.__version__} | CUDA {torch.version.cuda} | CPUs {os.cpu_count()}")

repo_url = "https://github.com/jaimejarauta/decoupled-delta-net.git"
branch = "feature/vla-knew"
target_dir = "/content/vla-knew"
if not os.path.exists(target_dir):
    subprocess.run(["git", "clone", "-b", branch, repo_url, target_dir], check=True)
else:
    subprocess.run(["git", "-C", target_dir, "fetch", "origin", branch], check=True)
    subprocess.run(["git", "-C", target_dir, "reset", "--hard", f"origin/{branch}"], check=True)
os.chdir(target_dir)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Step 4: Installs. Pinned OpenVLA deps (transformers 4.40.1) + robosuite 1.4.1 / mujoco 3.2.3 (as openpi's LIBERO env),
# LIBERO at a fixed commit (--no-deps: its requirements.txt pins 2022 versions), NVIDIA EGL vendor file for headless MuJoCo.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cmake"], check=False)  # hf-egl-probe builds with CMake (no py3.13 wheel)
_r = subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], capture_output=True, text=True)
if _r.returncode:  # Colab hides child-process output: print pip's real error before failing
    print(_r.stdout[-4000:]); print(_r.stderr[-6000:])
_r.check_returncode()
LIBERO_DIR, LIBERO_SHA = "/content/LIBERO", "8f1084e3132a39270c3a13ebe37270a43ece2a01"
if not os.path.exists(LIBERO_DIR):
    subprocess.run(["git", "clone", "-q", "https://github.com/Lifelong-Robot-Learning/LIBERO.git", LIBERO_DIR], check=True)
subprocess.run(["git", "-C", LIBERO_DIR, "checkout", "-q", LIBERO_SHA], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", LIBERO_DIR], check=True)
egl = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(egl):
    os.makedirs(os.path.dirname(egl), exist_ok=True)
    open(egl, "w").write('{"file_format_version" : "1.0.0", "ICD" : { "library_path" : "libEGL_nvidia.so.0" }}')
os.environ.update(MUJOCO_GL="egl", PYOPENGL_PLATFORM="egl", TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD="1", TOKENIZERS_PARALLELISM="false")
if os.environ.get("HF_TOKEN"):
    import huggingface_hub
    huggingface_hub.login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
# fresh interpreter, so it sees the new packages
print(subprocess.run([sys.executable, "-c", "import numpy, transformers, mujoco, robosuite, vk; vk.libero_setup(); "
                      "from libero.libero import benchmark; print('numpy', numpy.__version__, '| transformers', transformers.__version__, "
                      "'| mujoco', mujoco.__version__, '| libero ok')"], capture_output=True, text=True, check=True).stdout)

In [ ]:
# Step 5: GPU pre-flight (pytest, VK_PREFLIGHT=1): real LIBERO env + goal predicates, decode == predict_action,
# batched == single (>= 90% tokens), differentiable pixels == processor, patch gradient, env worker pool + mini group.
# ABORTS (and unassigns) if anything fails. First run also downloads the checkpoint into HF_HOME on Drive.
if subprocess.run([sys.executable, "run_all.py", "--preflight-only"]).returncode:
    raise RuntimeError("GPU pre-flight failed -> campaign aborted (see pytest output above)")
print("[Pre-flight] passed")

In [ ]:
# Step 6: Campaign (resumable): pilot -> frozen plan rule -> patch -> main rollouts -> gate analysis
p = subprocess.Popen([sys.executable, "-u", "run_all.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if "Loading checkpoint shards" not in line:
        print(line, end="")
if p.wait():
    raise RuntimeError(f"run_all.py failed (exit {p.returncode}); re-run the notebook to resume from Drive")

In [ ]:
# Step 7: Results
print(open(os.path.join(os.environ["VK_RESULTS"], "RESULTS.md")).read())

In [ ]:
# Step 8: Flush Drive & unassign GPU
auto_teardown()